In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import pickle
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('Housing.csv')

print("="*80)
print("HOUSE PRICE PREDICTION PROJECT")
print("="*80)
print(f"\nDataset Shape: {df.shape}")
print("\nFirst 5 rows:")
print(df.head())
print("\nColumn names:")
print(df.columns.tolist())

print("\n" + "="*80)
print("DATA TYPE CHECK")
print("="*80)
print("\nData types of each column:")
for col in df.columns:
    print(f"{col:20s}: {df[col].dtype}")

print("\n" + "="*80)
print("CATEGORICAL COLUMNS CHECK")
print("="*80)
object_cols = df.select_dtypes(include=['object']).columns
for col in object_cols:
    print(f"\n{col}:")
    print(f"  Unique values: {df[col].unique()}")
    print(f"  Value counts:")
    print(df[col].value_counts())

print("\n" + "="*80)
print("DATA PREPROCESSING")
print("="*80)

df_processed = df.copy()

binary_columns = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']

for col in binary_columns:
    if col in df_processed.columns:
        print(f"\nProcessing {col}:")
        print(f"  Before: {df_processed[col].unique()}")
        df_processed[col] = df_processed[col].astype(str).str.lower().str.strip()
        df_processed[col] = df_processed[col].map({'yes': 1, 'no': 0})
        print(f"  After: {df_processed[col].unique()}")
        print(f"  Data type: {df_processed[col].dtype}")

if 'furnishingstatus' in df_processed.columns:
    print(f"\nProcessing furnishingstatus:")
    print(f"  Before: {df_processed['furnishingstatus'].unique()}")
    df_processed['furnishingstatus'] = df_processed['furnishingstatus'].astype(str).str.lower().str.strip()
    furnishing_map = {'unfurnished': 0, 'semi-furnished': 1, 'furnished': 2}
    df_processed['furnishingstatus'] = df_processed['furnishingstatus'].map(furnishing_map)
    print(f"  After: {df_processed['furnishingstatus'].unique()}")

for col in df_processed.columns:
    if df_processed[col].dtype == 'object':
        print(f"\nAttempting to convert {col} to numeric...")
        try:
            df_processed[col] = pd.to_numeric(df_processed[col])
            print(f"  Successfully converted {col} to numeric")
        except:
            print(f"  Could not convert {col}, checking unique values...")
            print(f"    Unique: {df_processed[col].unique()[:10]}")

print("\n" + "="*80)
print("VERIFY ALL COLUMNS ARE NUMERIC")
print("="*80)
for col in df_processed.columns:
    print(f"{col:20s}: {df_processed[col].dtype}")

non_numeric_cols = df_processed.select_dtypes(include=['object']).columns.tolist()
if non_numeric_cols:
    print(f"\nWARNING: Still have non-numeric columns: {non_numeric_cols}")
    df_processed = df_processed.drop(columns=non_numeric_cols)
    print(f"Dropped columns: {non_numeric_cols}")

print("\n" + "="*80)
print("CREATING ADDITIONAL FEATURES")
print("="*80)

if 'price' in df_processed.columns and 'area' in df_processed.columns:
    df_processed['price_per_sqft'] = df_processed['price'] / df_processed['area']
    print("Created 'price_per_sqft'")

if 'bedrooms' in df_processed.columns and 'bathrooms' in df_processed.columns:
    df_processed['total_rooms'] = df_processed['bedrooms'] + df_processed['bathrooms']
    print("Created 'total_rooms'")
    
    df_processed['room_ratio'] = df_processed['bedrooms'] / df_processed['bathrooms'].replace(0, 1)
    print("Created 'room_ratio'")

luxury_cols = [col for col in ['airconditioning', 'hotwaterheating', 'guestroom', 'prefarea'] 
               if col in df_processed.columns]
if luxury_cols:
    df_processed['luxury_score'] = df_processed[luxury_cols].sum(axis=1)
    print(f"Created 'luxury_score' from {luxury_cols}")

if 'parking' in df_processed.columns and 'prefarea' in df_processed.columns:
    df_processed['parking_premium'] = df_processed['parking'] * df_processed['prefarea']
    print("Created 'parking_premium'")

if 'area' in df_processed.columns and 'bedrooms' in df_processed.columns:
    df_processed['area_per_bedroom'] = df_processed['area'] / df_processed['bedrooms'].replace(0, 1)
    print("Created 'area_per_bedroom'")

print("\n" + "="*80)
print("MODEL PREPARATION")
print("="*80)

target = 'price'

if target not in df_processed.columns:
    print(f"ERROR: '{target}' column not found!")
    print(f"Available: {df_processed.columns.tolist()}")
    exit()

exclude = [target, 'price_per_sqft']
exclude = [col for col in exclude if col in df_processed.columns]

feature_columns = [col for col in df_processed.columns if col not in exclude]

print(f"\nFeatures ({len(feature_columns)}):")
for i, col in enumerate(feature_columns, 1):
    print(f"  {i}. {col}")

X = df_processed[feature_columns].copy()
y = df_processed[target].copy()

print("\nFinal data type check:")
print(f"X shape: {X.shape}")
print(f"X dtypes: {X.dtypes.unique()}")

for col in X.columns:
    if X[col].dtype == 'object':
        print(f"Converting {col} to numeric...")
        X[col] = pd.to_numeric(X[col], errors='coerce')
        
if X.isnull().any().any():
    print("\nFilling NaN values with column means...")
    X = X.fillna(X.mean())
    
print(f"\nAll features are numeric now")
print(f"No missing values: {not X.isnull().any().any()}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nTraining set: {X_train_scaled.shape[0]} samples")
print(f"Testing set: {X_test_scaled.shape[0]} samples")

print("\n" + "="*80)
print("MODEL TRAINING")
print("="*80)

models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Lasso Regression': Lasso(alpha=0.01),
    'Decision Tree': DecisionTreeRegressor(max_depth=10, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'KNN (k=5)': KNeighborsRegressor(n_neighbors=5)
}

results = []
best_model = None
best_r2 = -np.inf

for name, model in models.items():
    try:
        print(f"\nTraining {name}...")
        model.fit(X_train_scaled, y_train)
        
        y_pred = model.predict(X_test_scaled)
        
        test_r2 = r2_score(y_test, y_pred)
        test_mae = mean_absolute_error(y_test, y_pred)
        test_mse = mean_squared_error(y_test, y_pred)
        
        results.append({
            'Model': name,
            'Test R²': test_r2,
            'Test MAE (₹)': f"₹{test_mae:,.0f}",
            'Test MSE': f"{test_mse:,.0f}"
        })
        
        print(f"  Test R²: {test_r2:.4f}")
        print(f"  Test MAE: ₹{test_mae:,.0f}")
        
        if test_r2 > best_r2:
            best_r2 = test_r2
            best_model = model
            best_model_name = name
    except Exception as e:
        print(f"  Failed: {str(e)}")

print("\n" + "="*80)
print("MODEL PERFORMANCE SUMMARY")
print("="*80)
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

print("\n" + "="*80)
print(f"BEST MODEL: {best_model_name}")
print(f"R² Score: {best_r2:.4f} ({best_r2*100:.1f}% variance explained)")
print("="*80)

if hasattr(best_model, 'feature_importances_'):
    print("\n" + "="*80)
    print("TOP 10 FEATURE IMPORTANCE")
    print("="*80)
    importance_df = pd.DataFrame({
        'Feature': feature_columns,
        'Importance': best_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    for i, row in importance_df.head(10).iterrows():
        print(f"{row['Feature']:25s}: {row['Importance']:.4f} ({row['Importance']*100:.1f}%)")

print("\n" + "="*80)
print("SAVING MODEL")
print("="*80)

with open('house_price_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

with open('feature_columns.pkl', 'wb') as f:
    pickle.dump(feature_columns, f)

print("Model saved successfully!")

def predict_price():
    print("\n" + "="*80)
    print("PREDICT HOUSE PRICE")
    print("="*80)
    
    input_data = {}
    for col in feature_columns:
        if col in ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']:
            val = input(f"{col} (yes/no): ").strip().lower()
            input_data[col] = 1 if val == 'yes' else 0
        elif col == 'furnishingstatus':
            val = input(f"{col} (unfurnished/semi-furnished/furnished): ").strip().lower()
            mapping = {'unfurnished': 0, 'semi-furnished': 1, 'furnished': 2}
            input_data[col] = mapping.get(val, 1)
        elif col in ['total_rooms', 'room_ratio', 'luxury_score', 'parking_premium', 'area_per_bedroom']:
            continue
        else:
            val = input(f"{col}: ")
            try:
                input_data[col] = float(val)
            except:
                input_data[col] = 0
    
    if 'total_rooms' in feature_columns and 'bedrooms' in input_data and 'bathrooms' in input_data:
        input_data['total_rooms'] = input_data['bedrooms'] + input_data['bathrooms']
    if 'room_ratio' in feature_columns and 'bedrooms' in input_data:
        input_data['room_ratio'] = input_data['bedrooms'] / max(input_data.get('bathrooms', 1), 1)
    if 'luxury_score' in feature_columns:
        luxury = ['airconditioning', 'hotwaterheating', 'guestroom', 'prefarea']
        input_data['luxury_score'] = sum(input_data.get(col, 0) for col in luxury if col in input_data)
    if 'parking_premium' in feature_columns:
        input_data['parking_premium'] = input_data.get('parking', 0) * input_data.get('prefarea', 0)
    if 'area_per_bedroom' in feature_columns:
        input_data['area_per_bedroom'] = input_data.get('area', 0) / max(input_data.get('bedrooms', 1), 1)
    
    input_df = pd.DataFrame([input_data])
    for col in feature_columns:
        if col not in input_df.columns:
            input_df[col] = 0
    input_df = input_df[feature_columns]
    
    scaled = scaler.transform(input_df)
    prediction = best_model.predict(scaled)[0]
    
    print("\n" + "="*80)
    print(f"PREDICTED PRICE: ₹{prediction:,.2f}")
    print(f"(₹{prediction/10000000:.2f} Crore)")
    print("="*80)
    return prediction

choice = input("\nWould you like to predict a house price? (yes/no): ").strip().lower()
if choice == 'yes':
    predict_price()

print("\nProject completed successfully!")

HOUSE PRICE PREDICTION PROJECT

Dataset Shape: (545, 13)

First 5 rows:
      price  area  bedrooms  bathrooms  stories mainroad guestroom basement  \
0  13300000  7420         4          2        3      yes        no       no   
1  12250000  8960         4          4        4      yes        no       no   
2  12250000  9960         3          2        2      yes        no      yes   
3  12215000  7500         4          2        2      yes        no      yes   
4  11410000  7420         4          1        2      yes       yes      yes   

  hotwaterheating airconditioning  parking prefarea furnishingstatus  
0              no             yes        2      yes        furnished  
1              no             yes        3       no        furnished  
2              no              no        2      yes   semi-furnished  
3              no             yes        3      yes        furnished  
4              no             yes        2       no        furnished  

Column names:
['price', 'a